In [1]:
pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
import datetime
import time
import os

print("Libraries installed and imported successfully.")

Libraries installed and imported successfully.


## 2. Initialize the Embedding Model and Vector Database

We'll use a pre-trained `all-MiniLM-L6-v2` model for generating text embeddings and set up a ChromaDB client.

In [3]:
# Initialize embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB client and collection
# We'll use a persistent client to store the data on disk
CHROMA_DB_PATH = "./chroma_db"
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collection_name = "chatbot_knowledge_base"
collection = client.get_or_create_collection(name=collection_name)

print(f"ChromaDB client initialized with collection: {collection_name}")
print(f"Data will be stored at: {os.path.abspath(CHROMA_DB_PATH)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB client initialized with collection: chatbot_knowledge_base
Data will be stored at: /content/chroma_db


In [4]:
def add_to_knowledge_base(new_texts):
    if not isinstance(new_texts, list):
        new_texts = [new_texts]

    # Generate embeddings for the new texts
    embeddings = model.encode(new_texts).tolist()

    # Generate unique IDs and metadata for each text
    current_time = datetime.datetime.now().isoformat()
    ids = [f"doc_{int(time.time() * 1000)}_{i}" for i in range(len(new_texts))]
    metadatas = [{'timestamp': current_time, 'source': 'dynamic_update'} for _ in new_texts]

    # Add to ChromaDB
    collection.add(
        embeddings=embeddings,
        documents=new_texts,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Added {len(new_texts)} new documents to the knowledge base.")

# Example usage:
new_info_1 = "The capital of France is Paris. It is famous for its Eiffel Tower."
add_to_knowledge_base(new_info_1)

new_info_2 = [
    "Chatbots are AI programs that simulate human conversation through voice or text commands.",
    "Machine learning is a subset of AI that enables systems to learn from data without explicit programming."
]
add_to_knowledge_base(new_info_2)

print(f"Current number of documents in knowledge base: {collection.count()}")

Added 1 new documents to the knowledge base.
Added 2 new documents to the knowledge base.
Current number of documents in knowledge base: 3


In [5]:
print("Simulating a delay before the next update...")
time.sleep(2) # Simulate some time passing

new_info_3 = "Artificial intelligence is a broad field of computer science that gives computers the ability to perform human-like cognitive functions."
add_to_knowledge_base(new_info_3)

print(f"Updated number of documents in knowledge base: {collection.count()}")

Simulating a delay before the next update...
Added 1 new documents to the knowledge base.
Updated number of documents in knowledge base: 4


In [9]:
import pandas as pd
import numpy as np

# Seed for reproducibility
np.random.seed(42)

# Number of projects
num_projects = 15

# Generate project names
project_names = [f'Project {i+1}' for i in range(num_projects)]

# Generate statuses
statuses = np.random.choice(['Completed', 'In Progress', 'On Hold', 'Canceled', 'Planned'], num_projects, p=[0.3, 0.4, 0.1, 0.05, 0.15])

# Generate start dates (within the last year)
start_dates = pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, num_projects), unit='D')

# Generate end dates (after start dates, some in the future)
# Initialize as a list for mutability
end_dates = (start_dates + pd.to_timedelta(np.random.randint(30, 365, num_projects), unit='D')).tolist()

# Adjust end dates for 'Completed' and 'Canceled' projects to be in the past
for i in range(num_projects):
    if statuses[i] == 'Completed':
        # Ensure completion within a reasonable timeframe in the past
        adjusted_end_date = start_dates[i] + pd.to_timedelta(np.random.randint(30, 200), unit='D')
        if adjusted_end_date > pd.Timestamp.now():
            adjusted_end_date = pd.Timestamp.now() - pd.to_timedelta(np.random.randint(1, 30), unit='D')
        end_dates[i] = adjusted_end_date

    elif statuses[i] == 'Canceled':
        # Canceled early
        adjusted_end_date = start_dates[i] + pd.to_timedelta(np.random.randint(10, 100), unit='D')
        if adjusted_end_date > pd.Timestamp.now():
            adjusted_end_date = pd.Timestamp.now() - pd.to_timedelta(np.random.randint(1, 30), unit='D')
        end_dates[i] = adjusted_end_date


# Generate budget (in thousands)
budgets = np.random.randint(50, 500, num_projects) * 1000

# Generate actual cost (relative to budget)
actual_costs = []
for i in range(num_projects):
    if statuses[i] == 'Completed':
        # Completed projects might be slightly over or under budget
        actual_costs.append(budgets[i] * np.random.uniform(0.9, 1.1))
    elif statuses[i] == 'Canceled':
        # Canceled projects have spent some portion of budget
        actual_costs.append(budgets[i] * np.random.uniform(0.1, 0.5))
    else:
        # In-progress/On-hold/Planned projects have spent less than budget currently
        actual_costs.append(budgets[i] * np.random.uniform(0.05, 0.7))
actual_costs = np.array(actual_costs)

# Generate progress percentage
progress_percentages = []
for i in range(num_projects):
    if statuses[i] == 'Completed':
        progress_percentages.append(100)
    elif statuses[i] == 'Canceled':
        progress_percentages.append(np.random.randint(5, 60))
    elif statuses[i] == 'Planned':
        progress_percentages.append(0)
    else: # In Progress, On Hold
        progress_percentages.append(np.random.randint(10, 95))
progress_percentages = np.array(progress_percentages)

# Generate priorities
priorities = np.random.choice(['High', 'Medium', 'Low'], num_projects, p=[0.3, 0.5, 0.2])

# Create DataFrame
project_data = pd.DataFrame({
    'Project_Name': project_names,
    'Status': statuses,
    'Start_Date': start_dates,
    'End_Date': end_dates, # Use the modified list
    'Budget': budgets,
    'Actual_Cost': actual_costs,
    'Progress_Percentage': progress_percentages,
    'Priority': priorities
})

# Convert date columns to datetime objects (end_dates is already datetime objects in list, but this ensures consistency)
project_data['Start_Date'] = pd.to_datetime(project_data['Start_Date'])
project_data['End_Date'] = pd.to_datetime(project_data['End_Date'])

# Display the first 5 rows and info
display(project_data.head())
project_data.info()

,Project_Name,Status,Start_Date,End_Date,Budget,Actual_Cost,Progress_Percentage,Priority
0,Project 1,In Progress,2023-10-04,2023-12-27,255000,87296.747606,38,Medium
1,Project 2,Planned,2023-06-10,2024-03-09,130000,39890.194942,0,High
2,Project 3,On Hold,2023-11-10,2024-10-24,469000,305941.955237,24,High
3,Project 4,In Progress,2023-01-22,2023-07-01,99000,51749.952933,54,Low
4,Project 5,Completed,2023-09-10,2024-01-07,409000,394811.034888,100,Medium


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Project_Name         15 non-null     object        
 1   Status               15 non-null     object        
 2   Start_Date           15 non-null     datetime64[ns]
 3   End_Date             15 non-null     datetime64[ns]
 4   Budget               15 non-null     int64         
 5   Actual_Cost          15 non-null     float64       
 6   Progress_Percentage  15 non-null     int64         
 7   Priority             15 non-null     object        
dtypes: datetime64[ns](2), float64(1), int64(2), object(3)
memory usage: 1.1+ KB


In [6]:
def query_knowledge_base(query_text, n_results=2):
    # Generate embedding for the query
    query_embedding = model.encode([query_text]).tolist()

    # Query the collection
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'distances', 'metadatas']
    )

    print(f"\nQuery: {query_text}")
    if results['documents'] and results['documents'][0]:
        for i, doc in enumerate(results['documents'][0]):
            print(f"--- Result {i+1} (Distance: {results['distances'][0][i]:.4f}) ---")
            print(f"Document: {doc}")
            print(f"Metadata: {results['metadatas'][0][i]}")
    else:
        print("No relevant documents found.")

# Test queries
query_knowledge_base("What is the capital of France?")
query_knowledge_base("What is machine learning?")
query_knowledge_base("What is AI?")
query_knowledge_base("Tell me about chatbots.")
query_knowledge_base("Something unrelated like the color of the sky")


Query: What is the capital of France?
--- Result 1 (Distance: 0.5951) ---
Document: The capital of France is Paris. It is famous for its Eiffel Tower.
Metadata: {'source': 'dynamic_update', 'timestamp': '2026-07-22T08:25:08.922510'}
--- Result 2 (Distance: 1.9062) ---
Document: Machine learning is a subset of AI that enables systems to learn from data without explicit programming.
Metadata: {'source': 'dynamic_update', 'timestamp': '2026-07-22T08:25:09.024350'}

Query: What is machine learning?
--- Result 1 (Distance: 0.5304) ---
Document: Machine learning is a subset of AI that enables systems to learn from data without explicit programming.
Metadata: {'timestamp': '2026-07-22T08:25:09.024350', 'source': 'dynamic_update'}
--- Result 2 (Distance: 1.0265) ---
Document: Artificial intelligence is a broad field of computer science that gives computers the ability to perform human-like cognitive functions.
Metadata: {'timestamp': '2026-07-22T08:25:11.713242', 'source': 'dynamic_update'}

